# Quality control: send random JSON via API key

This notebook sends a **random JSON payload** to the quality control **run** endpoint using the **X-API-Key** header.

- **Endpoint:** `POST /api/quality/run`
- **Auth:** Header `X-API-Key: <your-api-key>`
- **Body:** `{"data": [ ... ]}` — `data` must be a list of objects (e.g. rows of records).

Create an operation and get an API key from the Quality Control UI, then set `API_KEY` below (or use an env var).

In [4]:
import os
import random
import string
import json
import urllib.request

# Configure: base URL of the statsmed backend and your QC operation API key
API_BASE_URL = os.environ.get("STATSMED_API_URL", "http://localhost:8000")
API_KEY = "N_RJSFv_iBofblnbOkrxMb5vU0krD42VhsY_uo45Z0w"#os.environ.get("STATSMED_QC_API_KEY", "")  # set here or export STATSMED_QC_API_KEY

RUN_URL = f"{API_BASE_URL.rstrip('/')}/api/quality/run"

In [5]:
def random_json_list(n_rows=5, n_cols=3, key_prefix="col"):
    """Build a list of n_rows objects, each with n_cols random keys and values."""
    rows = []
    for _ in range(n_rows):
        row = {}
        for j in range(n_cols):
            k = f"{key_prefix}_{j}" if n_cols else f"{key_prefix}_{random.randint(0, 99)}"
            # mix of int, float, str, and occasional None
            choice = random.choice(["int", "float", "str", "null"])
            if choice == "int":
                row[k] = random.randint(-100, 100)
            elif choice == "float":
                row[k] = round(random.uniform(-10, 10), 2)
            elif choice == "str":
                row[k] = "".join(random.choices(string.ascii_letters, k=random.randint(2, 8)))
            else:
                row[k] = None
        rows.append(row)
    return rows

payload = {"data": random_json_list(n_rows=4, n_cols=3)}
print("Payload (data):")
print(json.dumps(payload, indent=2))

Payload (data):
{
  "data": [
    {
      "col_0": 75,
      "col_1": "JUDD",
      "col_2": -31
    },
    {
      "col_0": null,
      "col_1": 93,
      "col_2": -49
    },
    {
      "col_0": -8.93,
      "col_1": "qPE",
      "col_2": -6
    },
    {
      "col_0": "EovZqjux",
      "col_1": "PyO",
      "col_2": "yBgRMZ"
    }
  ]
}


In [6]:
if not API_KEY.strip():
    raise ValueError(
        "Set API_KEY in the cell above or export STATSMED_QC_API_KEY. "
        "Get the key from the Quality Control UI after creating an operation."
    )

body = json.dumps(payload).encode("utf-8")
req = urllib.request.Request(
    RUN_URL,
    data=body,
    method="POST",
    headers={
        "Content-Type": "application/json",
        "X-API-Key": API_KEY.strip(),
    },
)

with urllib.request.urlopen(req) as resp:
    result = json.loads(resp.read().decode())

print("Response:")
print(json.dumps(result, indent=2))

Response:
{
  "operation_id": 1,
  "operation_name": "Anforderungen",
  "success": false,
  "results": [
    {
      "name": "Unnamed",
      "function_type": "statsmed_test",
      "passed": false,
      "message": "Error: 'b'"
    },
    {
      "name": "Unnamed",
      "function_type": "statsmed_test",
      "passed": false,
      "message": "Error: 'z'"
    }
  ]
}
